In [ ]:
# 2
import numpy as np                                                      # anything with numbers
import pandas as pd                                                     # anything with tables
import xarray as xr                                                     # geospatial data
import matplotlib.pyplot as plt                                         # make visuals
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from workshop_utils import PROCESSED_DIR                                # PROCESSED_DIR is the data/processed/ directory, where the data we need is stored

print("Ready!")

## *PART III: Land Use and Land Use Change*

`AUTHOR COMMENT` Insert small introduction: how has Philippines population evolved the last 100 years? How has industrialization affected the way we as humans use land? Introduce topics such as expansion, the move from small agricultural families to large-scale monopolies and monocultures. Urban expansion, etc.

By now, you've already worked with Hansen et al.'s Global Forest Change dataset quite a bit. We've seen that it has tracked forest loss, year by year, since 2001. Let's look at it again.

### Where has forest been lost?

Earlier, you already visualized `negros_forest_change.nc` and its two variables: `treecover2000` (percent tree cover in 2000) and `lossyear` (the year each pixel first lost forest cover: 1 = 2001, ..., 23 = 2023, 0 = no loss detected ever):

<img src="images/forest_loss.png" alt="Hansen Forest Loss" style="width:60%">

For the next exercises, let's **reload the** `treecover` **and** `lossyear` variables from `negros_forest_change.nc`. We'll try something new: in one line, open a dataset and extract just one variable, to avoid keeping the whole dataset in computer memory for no reason.

**🖊️ Extract** the two variables from the file in one line:

In [ ]:
treecover = xr.open_dataset("../data/processed/day2/negros_forest_change.nc")["treecover2000"]      # ✏️ Hint: xr.open_dataset(...)["variable_you_want"]
lossyear = xr.open_dataset("../data/processed/day2/negros_forest_change.nc")["lossyear"]

# this opens the land sea mask and cuts it to the same extent as our tree cover data
lsm = xr.open_dataarray("../data/processed/land-sea-mask_0p00833333.nc").sel(
    lon=slice(float(treecover.longitude.min()), float(treecover.longitude.max())),
    lat=slice(float(treecover.latitude.max()), float(treecover.latitude.min())),
)

> 💬 **Discussion:** Do you see certain hotspots where forest was lost? **What do you think are the causes** for the lost forest on Negros?

...

Except for some limited clusters, the forest loss looks rather scattered throughout the island. Before zooming in to some case studies, let's put one overall number on the whole record, **🖊️ what percentage of the Negros forest has been lost in 2001-2023?**

In [ ]:
# ✏️ What was forest in 2000?
# Let's say that if > 25% of a pixel was covered with trees, it was forest
is_forest_2000 = treecover > 25     # Hint: the unit of treecover is already in percent, so you can just make it binary with treecover > 25

# make a True-False binary of tree cover lost since 2001
# ✏️ what value in lossyear means no tree cover was lost?
lost_trees = lossyear > 0   # Hint: it's unit is 'years since 2000', where e.g. a pixel with value '3' means tree cover in that pixel was lost in 2003

# lost_trees is also where cover was lost that was <= 25% (no forest)
# to get lost_forest, we need to 
# ✏️ combine lost_trees with is_forest_2000
lost_forest = lost_trees & is_forest_2000       # Hint: & means AND

# ✏️ Compute the lost Negros forest percentage
# Hint: use .sum() to count total pixels in both lost_forest and is_forest_2000
# Hint: then divide the two and multiply by 100 to get %
percent_lost = lost_forest.sum() / is_forest_2000.sum() * 100


print(f"Fraction of the 2000 Negros forest lost between 2001-2023: {float(percent_lost):.1f}%")

**3.5%** of Negros forest lost. In the sections below, we'll be working with some different cut-outs of the forest loss data. To easily recompute how much of the forest was lost in a given area, we can **🖊️ wrap the above into a function:**

In [ ]:
def forest_lost(treecover_data, loss_data):
    # ✏️ Exactly the same as above, but now for any treecover_data, loss_data passed to this function
    is_forest = treecover_data > 25
    lost_trees = loss_data > 0
    lost_forest = lost_trees & is_forest
    percent_lost = lost_forest.sum() / is_forest.sum() * 100
    
    print(f"{float(percent_lost):.1f}% of forest in data lost.")
    return percent_lost

`AUTHOR COMMENT` without code, just put it here: how many hectares is that? How many football (soccer) fields of forest has been lost on Negros alone?

> **Discussion:** 3.5% of Negros forest lost in about 20 years time. Do you think this is a lot, little, normal? Is it more than you expected? Less?
---

### Zooming in: Guimaras

**Guimaras** — the smaller island to the West of Bacolod — **seems to have quite a bit of tree cover that was lost**. Let's zoom in to the island. We'll be looking at **two case studies** in particular:

<img src="images/case_studies.png" alt="Guimaras tree cover and forest loss, with two zoom case studies marked" style="width:70%">

**🖊️ Let's quantify lost Guimaras forest!** With your new `forest_lost` function, you can easily do that now.

In [ ]:
# the coordinate box that cuts out Guimaras:
GUIMARAS_BBOX = dict(longitude=slice(122.46, 122.75),
                     latitude=slice(10.76, 10.40))

# Let's use 'tc' as abbreviation for tree cover
tc_guimaras = treecover.sel(**GUIMARAS_BBOX)        # 💡 **GUIMARAS_BBOX here gives the longitude=slice(...) and latitude=(...) straight to the .sel(...) function.
lost_guimaras = lost_trees.sel(**GUIMARAS_BBOX)

# ✏️ Use your forest_loss() function to instantly print the percentage of forest lost in Guimaras
lost_Guimaras = forest_lost(tc_guimaras, lost_guimaras)

> **Discussion:** In Guimaras, 5.9% of forest was lost. That's quite a bit higher than the overall 3.5% we saw in the box containing both Negros and Guimaras. **Are there specific reasons for this?**

Let's zoom in to the two **case studies**, beginning with, again, using your `forest_lost` function to directly quantify the lost forest in those regions:

In [ ]:
CASE_1 = dict(longitude=slice(122.63, 122.65),
              latitude=slice(10.64, 10.625))

# ✏️ Cut the case 1 box out of the treecover data
tc_case1 = treecover.sel(**CASE_1)      # Hint: you can specify the coordinates, or use the same **CASE_1 trick as above

# ✏️ Cut the case 1 box out of the lost_trees data
lost_case1 = lost_trees.sel(**CASE_1)

# ✏️ use your forest_loss() function to instantly print the percentage of forest lost in case 1
fl_case1 = forest_lost(tc_case1, lost_case1)

CASE_2 = dict(longitude=slice(122.585, 122.595),
                    latitude=slice(10.6525, 10.64))

# ✏️ Cut the case 2 box out of the treecover data
tc_case2 = treecover.sel(**CASE_2)

# ✏️ Cut the case 2 box out of the lost_trees data
lost_case2 = lost_trees.sel(**CASE_2)

# ✏️ use your forest_loss() function to instantly print the percentage of forest lost in case 2
fl_case2 = forest_lost(tc_case2, lost_case2)

`AUTHOR COMMENT` Again, mention how many football fields that is. Do a discussion on why, in certain areas, so much of the forest has been lost on such a short time.

### Can we see it in satellite imagery?

Let's zoom all the way in on **case study 1**: `guimaras_zoom_imagery.nc` is a high-resolution crop of Landsat 5 and Sentinel-2 satellite imagery, specifically to that small patch. Note that while Sentinel-2 has 10m resolution, Landsat 5 only has 30m resolution. Here is what is looks like. Left: Landsat 5, middle: Sentinel-2, right: a screenshot from Google Maps.

`AUTHOR COMMENT`: As we've already delineated Case study 1 and 2 in a map above, we can drop the lower left subplot here. Convert it into a 1x3: Landsat - Sentinel - Google Maps

<img src="images/guimaras_zoom_comparison.png" alt="Zooming in on the loss cluster, true colour" style="width:70%">

What about **case study 2**, can we spot from our satellite imagery, by eye, where forest has been lost?

`AUTHOR COMMENT`: insert here a similar 1x3 subplot. Imagine still has to be created and written to `images/`. I made a Google Maps screenshot and put it under images/ as case_2_screenshot.png.

> 💬 **Discussion:** Can you confidently point to *where* forest was cleared, just from these panels?

**Clearly**, from true-color images alone, especially if resolution is limited, it's **very hard to distinguish** exactly where forest has been lost. But next to the passive `visual_r`, `visual_g`, and `visual_b` bands, the satellite imagery also contains active `red` and `nir` measurements, remember?

---

### Which satellite band shows forest cover best?

Let's zoom back out to all of Negros and look at the raw ingredients: `negros_imagery.nc`'s individual colour + infrared bands, next to Hansen's own tree-cover map.

In [ ]:
negros_img = xr.open_dataset(PROCESSED_DIR / "day2" / "negros_imagery.nc")
recent = negros_img.sel(time="2023-03-15")

treecover_negros = treecover.sel(longitude=slice(float(negros_img.longitude.min()), float(negros_img.longitude.max())),
                                  latitude=slice(float(negros_img.latitude.max()), float(negros_img.latitude.min())))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

treecover_negros.plot(ax=axes[0], cmap="Greens", vmin=0, vmax=100, add_colorbar=False)
axes[0].set_title("Tree cover (2000)")

for ax, band, cmap in zip(axes[1:4], ["visual_r", "visual_g", "visual_b"], ["Reds", "Greens", "Blues"]):
    recent[band].plot(ax=ax, cmap=cmap, vmin=0, vmax=255, add_colorbar=False)
    ax.set_title(band)

for ax, band in zip(axes[4:], ["red", "nir"]):
    recent[band].plot(ax=ax, cmap="viridis", vmin=0, vmax=0.5, add_colorbar=False)
    ax.set_title(band)

for ax in axes:
    ax.set_aspect("equal")

> 💬 **Discussion:** Does any single band look like a forest-cover map on its own? What band comes closest?

---

### NDVI: turning two bands into one meaningful number

**Healthy, lush-green vegetation absorbs a lot of red light but reflects a lot of near-infrared**. We can combine the `red` and `nir` bands in the [Normalized Difference Vegetation Index NDVI](https://en.wikipedia.org/wiki/Normalized_difference_vegetation_index), the most widely used metric to derive vegetation status from remote sensing data. NDVI is computed as:

**NDVI = (NIR − Red) / (NIR + Red)**

NDVI ranges from -1 to +1: strongly **negative** for water, **near zero** for bare soil/urban surfaces, and **high** (often >0.3) for healthy vegetation. Let's write a small reusable function for it:

In [ ]:
def compute_ndvi(nir, red):
    return (nir - red) / (nir + red)

# If you got that right, this should be True:
compute_ndvi(10, 5) == 1/3

Yay! Super simple function, but look what we can do with it:

In [ ]:
# --- Apply it to the 2023 (recent) composite and map it ---
ndvi_2023 = compute_ndvi(recent["nir"], recent["red"])

fig, ax = plt.subplots(figsize=(7, 7))
ndvi_2023.plot(ax=ax, cmap="RdYlGn", vmin=-1, vmax=1, cbar_kwargs={"label": "NDVI"})
ax.set_title("NDVI — 2023 composite")
ax.set_aspect("equal")

> **Discussion:** What do you see? What patterns are green, yellow, and red? Is the NDVI metric showing a different pattern than what can be observed with any individual band? Is it adding information?

**🖊️ Let's visualize** the forest cover, next to **different** maps of where **NDVI** exceeds a certain **threshold**:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

treecover_negros.plot(ax=axes[0], cmap="Greens", vmin=0, vmax=100, add_colorbar=False)
axes[0].set_title("Tree cover (2000)")

for ax, ndvi in zip(axes[1:], [0.5, 0.7, 0.8]):
    ndvi_2023.where(ndvi_2023 > ndvi).plot(cmap="Greens", vmin=0, vmax=ndvi, add_colorbar=False, ax=ax)
    ax.set_title(f"NDVI > {ndvi}")

for ax in axes:
    ax.contourf(lsm.lon, lsm.lat, lsm.values, levels=[0.75, 1.25], colors=["#666666"], alpha=0.3)
    ax.set_aspect("equal")

> **Discussion:** Cool, right? NDVI is clearly picking up a signal that is related to forest. This is what we expected, as it essentially quantifies whether vegetation is lush green, which forests tend to be. **Are there still differences between NDVI and the tree cover map?** Why might that be?

### From NDVI to land-cover classes

Let's turn NDVI into actual land-cover classes using a simple, interpretable rule — deliberately similar in spirit to Köppen's group logic: a handful of threshold conditions on physically meaningful numbers.

Looking at real values in the Negros composite above: NDVI ranges roughly -0.56 to +0.91 in both the 1994 and 2023 composites (mean ~0.31-0.32 — the two dates land in a very similar range this time). *Brightness* (the average of red and nir) is much lower over water than over land or bare ground. `negros_imagery.nc` stores physical **reflectance** (0-1 float), not raw digital numbers — so our brightness threshold has to live on that scale too. That gives us three classes:

| Class | Rule |
|---|---|
| **Water** | low brightness *and* low NDVI (dark in both bands — water absorbs rather than reflects) |
| **Vegetation** | high NDVI |
| **Bare/Urban** | everything else (moderate-to-high brightness, low NDVI) |

We'll use `brightness < 0.08` and `NDVI < 0.1` for Water, and `NDVI > 0.5` for Vegetation — thresholds picked by eye against the Negros scene's own numbers.

### 🖊️ Your turn: `classify_landcover_2D`

Reuse the exact "boolean-mask, reverse-priority overwrite" trick from `classify_koppen_2D` in Part I — same technique, brand new domain, xarray structure preserved throughout.

In [ ]:
def classify_landcover_2D(red, nir,
                          vegetation_thr=0.5,
                          brightness_thr=0.08):

    ndvi = compute_ndvi(nir, red).clip(-1, 1)  # a few pixels have near-zero red+nir -> guard the ratio
    brightness = (red + nir) / 2

    landcover = xr.full_like(red, "Bare/Urban", dtype="<U10")               # ✏️ default/"else" case
    landcover.values[(brightness < brightness_thr).values & (ndvi < 0.1).values] = "Water"   # ✏️ dark in both bands
    landcover.values[(ndvi > vegetation_thr).values] = "Vegetation"                               # ✏️ high NDVI, checked last -> wins
    landcover.values[np.isnan(red.values)] = ""                                        # ✏️ cloud/no-data gaps - not a real class

    return landcover


# --- Try it on the 2023 composite ---
# `AUTHOR COMMENT` What's printed here is unreadable to students without experience. Make it more readable by explicitly printing e.g. Bare/Urban: ... pixels, Vegetation: ... pixels, etc.
landcover_2023 = classify_landcover_2D(recent["red"], recent["nir"])
print(dict(zip(*np.unique(landcover_2023.values, return_counts=True))))

Now let's map it, using its own small categorical colour scheme — same recipe as the Köppen-Geiger maps:

In [ ]:
LC_INFO = [
    (1, "Water",      "#3B4CC0"),
    (2, "Vegetation", "#2E8B57"),
    (3, "Bare/Urban", "#C2A878"),
]
lc_cmap = mcolors.ListedColormap([c for _, _, c in LC_INFO])
lc_norm = mcolors.BoundaryNorm(boundaries=np.arange(0.5, len(LC_INFO) + 1.5), ncolors=len(LC_INFO))
LC_TO_CODE = {name: code for code, name, _ in LC_INFO}


def landcover_to_code(landcover):
    # 💡 same trick as class_to_code in Part I: map each string label to its numeric code, keep the DataArray/coords intact
    code_grid = xr.full_like(landcover, np.nan, dtype="float64")
    for name, code in LC_TO_CODE.items():
        code_grid.values[(landcover == name).values] = code
    return code_grid


fig, ax = plt.subplots(figsize=(7, 7))
code_2023 = landcover_to_code(landcover_2023)
ax.pcolormesh(code_2023.longitude, code_2023.latitude, code_2023, cmap=lc_cmap, norm=lc_norm)
ax.scatter(122.95, 10.68, color="black", s=40, zorder=5)
ax.set_title("Homemade land cover — 2023 composite")
ax.set_aspect("equal")
patches = [mpatches.Patch(color=color, label=name) for _, name, color in LC_INFO]
ax.legend(handles=patches, loc="lower left", frameon=True, fontsize=9)
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Compare this to the visual bands above — does Water/Vegetation/Bare-Urban line up with what you saw by eye?

Let's also sanity-check this against `treecover2000` — side by side, purely by eye (different sensor, different resolution, so no pixel-by-pixel score, same design choice as Part I Step 7):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

axes[0].pcolormesh(code_2023.longitude, code_2023.latitude, code_2023, cmap=lc_cmap, norm=lc_norm)
axes[0].set_title("Our homemade land cover (2023 composite)")

treecover_negros.plot(ax=axes[1], cmap="Greens", vmin=0, vmax=100, add_colorbar=False)
axes[1].set_title("Hansen tree cover (2000)")

for ax in axes:
    ax.set_aspect("equal")

### Zooming back in: classifying Guimaras at full resolution

`classify_landcover_2D` doesn't care what image you feed it — let's reuse it, unchanged, on the
high-resolution `guimaras_zoom_imagery.nc` crop from Section 3, starting with just the Sentinel-2
(recent) slice:

In [ ]:
zoom_img = xr.open_dataset(PROCESSED_DIR / "day2" / "guimaras_zoom_imagery.nc")
ZOOM_BBOX = dict(longitude=slice(122.63, 122.65), latitude=slice(10.64, 10.625))   # ✏️ the densest part of the loss cluster above


def to_rgb(ds_slice):
    r = np.nan_to_num(ds_slice["visual_r"].values)
    g = np.nan_to_num(ds_slice["visual_g"].values)
    b = np.nan_to_num(ds_slice["visual_b"].values)
    return np.dstack([r, g, b]).astype(np.uint8)


hist_zoom = zoom_img.sel(time="1994-03-15")
rec_zoom = zoom_img.sel(time="2024-03-15")   # ✏️ note: 2024, not 2023 -- Guimaras' own scene pool landed on a different median year than Negros'

`AUTHOR COMMENT` Earlier, we saw that an NDVI threshold of 0.7 approximated the forest cover map quite well. We can use that below to search for forest loss!

In [ ]:
landcover_zoom_s2 = classify_landcover_2D(rec_zoom["red"], rec_zoom["nir"], vegetation_thr=0.7, brightness_thr=0.08)
vals, counts = np.unique(landcover_zoom_s2.values, return_counts=True)
print({v: f"{c / landcover_zoom_s2.size:.1%}" for v, c in zip(vals, counts)})

code_zoom_s2 = landcover_to_code(landcover_zoom_s2)

fig, axes = plt.subplots(1, 2, figsize=(11, 6))
axes[0].imshow(to_rgb(rec_zoom))
axes[0].set_title("Sentinel-2 true colour (~2024)")
axes[0].axis("off")

axes[1].pcolormesh(code_zoom_s2.longitude, code_zoom_s2.latitude, code_zoom_s2, cmap=lc_cmap, norm=lc_norm)
axes[1].set_title("Our homemade land cover")
axes[1].set_aspect("equal")
patches = [mpatches.Patch(color=color, label=name) for _, name, color in LC_INFO]
axes[1].legend(handles=patches, loc="lower left", frameon=True, fontsize=8)
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Does it look good? Compare the two panels directly — does the classifier's Vegetation/Bare-Urban boundary line up with what you can see by eye in the true-colour image?
>
> Notice there's essentially no Water here (this crop is an inland forest hotspot, not coastline) and virtually everything classifies as Vegetation — 99.0%. That's not a bug: our 3-class rule has no way to tell thriving old-growth forest apart from land that was cleared and is now regrowing as scrub or cropland. Both look "green enough" to pass the NDVI > 0.3 bar. That gap is exactly why the Land Use Change section ahead uses a *stricter* NDVI threshold specifically for "forest."

---

### Landsat versus Sentinel land-cover classes

Same classifier, same crop, but now the *historical* (Landsat 5) slice too — does the same simple rule hold up switching sensors, the way it held up switching location?

In [ ]:
landcover_zoom_l5 = classify_landcover_2D(hist_zoom["red"], hist_zoom["nir"], vegetation_thr=0.7, brightness_thr=0.08)
vals, counts = np.unique(landcover_zoom_l5.values, return_counts=True)
print({v: f"{c / landcover_zoom_l5.size:.1%}" for v, c in zip(vals, counts)})

code_zoom_l5 = landcover_to_code(landcover_zoom_l5)

fig, axes = plt.subplots(1, 2, figsize=(11, 6))
for ax, code_grid, title in zip(axes, [code_zoom_l5, code_zoom_s2], ["Landsat 5 (~1994)", "Sentinel-2 (~2024)"]):
    ax.pcolormesh(code_grid.longitude, code_grid.latitude, code_grid, cmap=lc_cmap, norm=lc_norm)
    ax.set_title(title)
    ax.set_aspect("equal")

patches = [mpatches.Patch(color=color, label=name) for _, name, color in LC_INFO]
fig.legend(handles=patches, loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.suptitle("Homemade land cover -- Landsat vs. Sentinel, same crop")
plt.tight_layout(rect=[0, 0.06, 1, 1])

> 💬 **Discussion:** Uh-oh! Wait, was Hansen tree cover lying, did Guimaras actually do a massive afforestiation project rather than deforestation? Or is something else happening? **Why do we see such large differences between there two maps?**

If you look back at the case study satellite images, you'll notice something: the **Landsat 5 and Sentinel-2 imagery have very different color hues**. Large, systematic differences like this are a **red flag** that the two composites need calibrating onto a common scale before comparing them directly — exactly the check we'll do next, at full-island scale, in Land Use Change.

---

You just:
- Learned what NDVI is and why it works (red absorption vs. NIR reflection in healthy vegetation)
- Reused Part I's exact "rule -> vectorized function -> map" pattern in a brand new domain
- Classified real satellite imagery into land-cover classes, first for all of Negros, then at full 10 m resolution over a Guimaras forest-loss hotspot
- Compared classifications from two genuinely different sensors (Landsat 5, Sentinel-2) on the same patch of ground and found them in close agreement

## Land Use Change

We opened this Part with Hansen's own annual forest-loss labels; now let's cross-check that against change detected from *our own* homemade land-cover classifier — the same way Part I cross-checked its homemade climate map against Beck et al.

**One honest caveat:** these aren't the same window. Sentinel-2 only launched in 2015, so reaching back to ~1994 for imagery means switching sensors entirely — our composites span roughly 1994-2023, not Hansen's 2001-2023. We're not measuring the identical thing twice; we're making two independently-engineered measurements and seeing whether the two stories broadly agree.

### A data-quality check, revisited

Earlier versions of this composite pipeline had a real problem: Landsat 5's cloud mask false-positives heavily over open water (sunglint, whitecaps, low signal-to-noise on the older sensor), so a lot of genuinely clear ocean got thrown out as "cloud," leaving big gaps. The current `negros_imagery.nc` pools far more scenes per period than earlier attempts — let's check whether that actually fixed it, for both land and ocean:

In [ ]:
lsm_on_negros = lsm.rename({"lat": "latitude", "lon": "longitude"}).interp(
    latitude=negros_img.latitude, longitude=negros_img.longitude, method="nearest"
)
ocean_negros = lsm_on_negros == 0
land_negros = lsm_on_negros == 1

for label, t in [("1994 (historical)", "1994-03-15"), ("2023 (recent)", "2023-03-15")]:
    red = negros_img["red"].sel(time=t)
    nan_land = float(np.isnan(red.values)[land_negros.values].mean())
    nan_ocean = float(np.isnan(red.values)[ocean_negros.values].mean())
    print(f"{label}: {nan_land:.1%} no-data over LAND,   {nan_ocean:.1%} no-data over OCEAN")

> 💬 **Discussion:** Both composites are now essentially complete, over land *and* ocean — the missing-ocean-data problem from earlier pipeline attempts is gone (pooling far more scenes per period fixed it). That's genuinely useful here: complete, stable ocean pixels mean we can now use the ocean itself as a **calibration reference** between the two sensors, since open-ocean reflectance shouldn't systematically change over 30 years the way land cover does.

### Calibrating Landsat against Sentinel-2

Sensor differences (calibration, atmospheric correction, spectral response) can shift a whole composite's brightness up or down even when nothing on the ground changed. Since ocean reflectance should be roughly stable over time, any systematic difference between the historical and recent composite's *ocean* pixels is sensor bias, not real change. Sensor gain differences are typically **multiplicative** (a target twice as bright returns roughly twice the signal), so we calibrate with a ratio, not a fixed offset:

In [ ]:
hist_ocean_red = float(negros_img["red"].sel(time="1994-03-15").where(ocean_negros).mean())
hist_ocean_nir = float(negros_img["nir"].sel(time="1994-03-15").where(ocean_negros).mean())
rec_ocean_red = float(negros_img["red"].sel(time="2023-03-15").where(ocean_negros).mean())
rec_ocean_nir = float(negros_img["nir"].sel(time="2023-03-15").where(ocean_negros).mean())

red_ratio = rec_ocean_red / hist_ocean_red   # ✏️ how much brighter/darker Sentinel reads vs Landsat, over the same (stable) ocean
nir_ratio = rec_ocean_nir / hist_ocean_nir

print(f"Ocean-reference red:  historical={hist_ocean_red:.4f}  recent={rec_ocean_red:.4f}  ratio={red_ratio:.2f}")
print(f"Ocean-reference nir:  historical={hist_ocean_nir:.4f}  recent={rec_ocean_nir:.4f}  ratio={nir_ratio:.2f}")

> 💬 **Discussion:** Sentinel-2 reads consistently brighter than Landsat 5 over the same (supposedly stable) ocean — roughly 1.8-1.9x in both bands. That's a real, substantial sensor difference we'd otherwise risk mistaking for land-cover change. We'll rescale every historical pixel by these ratios before comparing it to the recent composite.

### Land-cover mapping on Guimaras, calibrated

Now let's bring in `guimaras_imagery.nc` — the whole island at 30 m, red/nir only (no RGB bands here, since we're doing NDVI-based classification, not eyeballing true colour). Apply the calibration, then use a stricter `NDVI > 0.7` threshold to define "forest" specifically — the broader `NDVI > 0.3` "Vegetation" class from before can't distinguish real forest from regrowing scrub (Section 4's discussion) — and compare the resulting loss fraction to Hansen's own number for Guimaras from Section 2.

In [ ]:
guimaras_img = xr.open_dataset(PROCESSED_DIR / "day2" / "guimaras_imagery.nc")
hist_g = guimaras_img.sel(time="1994-03-15")
rec_g = guimaras_img.sel(time="2024-03-15")

FOREST_NDVI_THR = 0.7   # ✏️ stricter than the 0.3 "Vegetation" cutoff -- isolates dense/healthy canopy specifically
ndvi_rec_g = compute_ndvi(rec_g["red"], rec_g["nir"])
forest_rec_g = ndvi_rec_g > FOREST_NDVI_THR

# --- Uncalibrated, for comparison ---
ndvi_hist_raw = compute_ndvi(hist_g["red"], hist_g["nir"])
forest_hist_raw = ndvi_hist_raw > FOREST_NDVI_THR
lost_raw = forest_hist_raw & ~forest_rec_g
frac_lost_raw = float(lost_raw.sum()) / float(forest_hist_raw.sum())

# --- Calibrated ---
hist_red_cal = hist_g["red"] * red_ratio
hist_nir_cal = hist_g["nir"] * nir_ratio
ndvi_hist_g = compute_ndvi(hist_red_cal, hist_nir_cal)
forest_hist_g = ndvi_hist_g > FOREST_NDVI_THR
lost_forest_g = forest_hist_g & ~forest_rec_g
frac_lost_g = float(lost_forest_g.sum()) / float(forest_hist_g.sum())

print(f"Guimaras -- uncalibrated forest loss:  {frac_lost_raw * 100:.1f}%")
print(f"Guimaras -- calibrated forest loss:    {frac_lost_g * 100:.1f}%")
print(f"Guimaras -- Hansen forest loss, land-only (from Section 2): {loss_guimaras_land * 100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

axes[0].pcolormesh(lost_forest_g.longitude, lost_forest_g.latitude, lost_forest_g.where(lost_forest_g),
                    cmap=mcolors.ListedColormap(["firebrick"]))
axes[0].set_title(f"Our classifier: forest loss (NDVI>{FOREST_NDVI_THR}, calibrated)")

axes[1].pcolormesh(lost_guimaras.longitude, lost_guimaras.latitude, lost_guimaras.where(lost_guimaras),
                    cmap=mcolors.ListedColormap(["firebrick"]))
axes[1].set_title("Hansen forest loss (2001-2023)")

for ax in axes:
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Calibration moves the estimate in the right direction — uncalibrated forest loss comes out at **14.2%**, calibrated at **11.5%**, versus Hansen's independent **3.1%** — but it doesn't fully close the gap. A single global brightness ratio corrects the big, systematic sensor bias, but real cross-sensor differences (spectral response, atmospheric correction, spatial resolution) are more complex than one number per band can capture.
>
> Worth noticing too: our calibrated 1994 "forest" fraction (NDVI > 0.7) is only ~5%, well below Hansen's ~23% tree-cover baseline for Guimaras from Section 2. That's not necessarily a contradiction — `NDVI > 0.7` is a strict "dense, healthy canopy only" bar, stricter than Hansen's own tree-cover threshold, so a smaller number here is measuring something narrower, not necessarily something wrong. Real-world sensor calibration is genuinely hard; a good-faith attempt that improves things without perfectly matching an independent product is a realistic, honest outcome — not a failure.

### Culminate: land-use change across all of Negros

Let's finish where Part III started — but now with our own homemade classifier, at Negros-wide scale, using `negros_imagery.nc` and the exact same rule → function → grid → map pattern as before. `landcover_2023` was already computed back in Step 6 — let's classify the 1994 slice too and compare:

In [ ]:
historical = negros_img.sel(time="1994-03-15")
landcover_1994 = classify_landcover_2D(historical["red"], historical["nir"])
# landcover_2023 already computed in Step 6

for label, lc in [("1994", landcover_1994), ("2023", landcover_2023)]:
    vals, counts = np.unique(lc.values, return_counts=True)
    print(label, {v: f"{c / lc.size:.1%}" for v, c in zip(vals, counts)})

In [ ]:
def detect_change(before, after, from_class=None, to_class=None):
    # ✏️ from_class/to_class are optional: None means "any class". A pixel counts as changed if
    # it was `from_class` before, is `to_class` after, and the two dates actually differ.
    has_data  = (before != "") & (after != "")            # ✏️ ignore cloud/no-data gaps in either year!
    was_from  = True if from_class is None else (before == from_class)
    became_to = True if to_class   is None else (after == to_class)
    return has_data & was_from & became_to & (before != after)


lost_vegetation = detect_change(landcover_1994, landcover_2023, from_class="Vegetation")
was_vegetation_1994 = (landcover_1994 == "Vegetation")
print(f"Pixels that were Vegetation in 1994 but aren't anymore: {int(lost_vegetation.sum())}")

💡 Note: without the `has_data` check, cloud/no-data gaps in either year would get counted as spurious "change" — unlike the Guimaras-only calibration step above, we don't need a land-only mask here: both composites are complete over land *and* ocean.

Map the lost-vegetation mask, and check the overall fraction against Hansen's own number from the start of this Part:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
ax.pcolormesh(lost_vegetation.longitude, lost_vegetation.latitude, lost_vegetation.where(lost_vegetation),
              cmap=mcolors.ListedColormap(["firebrick"]))
ax.set_title("Lost vegetation, 1994 -> 2023 (our own classifier, all of Negros)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

frac_lost_ours = float(lost_vegetation.sum()) / float(was_vegetation_1994.sum())
print(f"Fraction of 1994's Vegetation-classified pixels that changed by 2023: {frac_lost_ours * 100:.1f}%")
print(f"For comparison -- Hansen's independent full 2001-2023 forest-loss record (from the start of this Part):")
print(f"  Our classifier: {frac_lost_ours * 100:.1f}%   Hansen (2001-2023, vs. 2000 baseline): {frac_lost_hansen_full * 100:.1f}%")

> 💬 **Discussion:** Our own classifier says roughly **2.9%** of 1994's vegetated pixels changed by 2023 — very close to Hansen's independent **~3.5%** forest-loss estimate over a comparable era. They're not identical: our "Vegetation" class lumps forest together with farmland/grass, while Hansen's `lossyear` specifically tracks forest, and the two records span slightly different windows. But landing within the same ballpark, with a much simpler rule and no per-pixel training data, is exactly what properly-engineered composite data and a robust (NDVI-based) rule buys you.

One last check — does Guimaras, the hotspot we opened with, actually show up as a hotspot in *our own* classifier too?

In [ ]:
lost_vegetation_guimaras = lost_vegetation.sel(**GUIMARAS_BBOX)
was_vegetation_guimaras = was_vegetation_1994.sel(**GUIMARAS_BBOX)
frac_lost_guimaras_ours = float(lost_vegetation_guimaras.sum()) / float(was_vegetation_guimaras.sum())

print(f"Guimaras (our classifier):     {frac_lost_guimaras_ours * 100:.1f}% vegetation loss")
print(f"Whole domain (our classifier): {frac_lost_ours * 100:.1f}% vegetation loss")

> 💬 **Discussion:** Guimaras comes out above the domain-wide average in our own classifier too — **4.1%** vegetation loss versus **2.9%** for all of Negros. Not as dramatic a gap as Hansen's forest-specific number (since our "Vegetation" class is coarser than "forest"), but the same direction, from a completely independent method. Two different datasets, two different definitions of "vegetation," and both point at the same island.

---

> 💬 **Final discussion:**
> - Look back at Part II's Philippine Eagle map and the EN/CR/VU mammal species you found on Negros. Does the forest-loss pattern from Guimaras and the rest of the island help explain why those species are threatened?
> - Across all of Day 2, you built **two** homemade classifiers from raw data — one for climate (Part I), one for land cover (Part III) — validated both against independent "official" products (Beck et al., Hansen), and used the second one to directly measure real-world change over time. That loop (rule -> function -> grid -> map -> validate) is the core pattern behind an enormous amount of real environmental data science.

---

## Well done — Day 2 complete!

Across today you:
- Learned `def` functions and `for` loops (including nested loops), and used them to build a full climate classification pipeline from scratch
- Classified the entire Philippines at three points in time, and validated your homemade model against Beck et al. (2023)
- Explored forest cover, satellite imagery, and real (messy!) biodiversity data for Negros
- Used Hansen's forest-loss data to find a real, concentrated deforestation hotspot (Guimaras) and let it motivate building a land-cover classifier, rather than the other way around
- Built a second homemade classifier — this time for land cover — reusing the exact same rule -> function -> grid -> map pattern from Part I
- Zoomed all the way in to full sensor resolution on a real forest-loss hotspot, and compared classifications from two different satellite sensors on the same ground
- Calibrated Landsat against Sentinel-2 using stable ocean reflectance as a reference, and saw honestly how far a simple global correction does (and doesn't) go
- Measured real land-use change across all of Negros and Guimaras using your own classifier, cross-checked against Hansen et al.'s independent forest-loss product
- Connected climate, vegetation, satellite imagery, biodiversity, and land-use change into a single, coherent story about these islands

Tomorrow: Day 3.